<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

# GUU — Task 4: Parent–Subsidiary Relationships (Exhibit 21)

This notebook implements the first stage of a parent–subsidiary pipeline using SEC 10-K Exhibit 21.

**Goal (Stage 1):**  
For each parent company in our normalized company list, attempt to:
1. Fetch the most recent 10-K filing.
2. Locate Exhibit 21 (List of Subsidiaries).
3. Extract the names of listed subsidiaries.
4. Store the relationships in a standardized table:

- `parent_name`
- `parent_cik`
- `subsidiary_name_raw`
- `subsidiary_name_clean`
- `source_type` (for now: `"exhibit_21"`)
- `source_url`
- `confidence`
- `extraction_notes`

This is a **seed, high-precision** dataset; coverage will be limited and later expanded with additional methods (e.g., Item 1 text, heuristics, FEC matching).


# Imports and basic configuration

In [1]:
import pandas as pd
import requests
import time
import os
from rapidfuzz import process, fuzz
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import re


# IMPORTANT: Set a proper User-Agent per SEC guidelines
SEC_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "Chrome/120.0 DavoAcevedo/1.0 (davoacevedo@arizona.edu)"
}
# Optional: If you hit HTML archives (Archives/edgar), they may not require Host header
HTML_HEADERS = {
    "User-Agent": "Davo Acevedo-Cardona davoacevedo@arizona.edu",
}


# File paths and schema

In [2]:
# Input: normalized company list (already used in previous GUU tasks)
COMPANY_FILE = "unique_companies.csv" 

# Output: parent–subsidiary links (seed from Exhibit 21 only)
OUTPUT_FILE = "parent_subsidiaries_ex21_seed.csv"

# Input: contact-level file from Task 3
CONTACTS_FILE = r"C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 4\company_contacts_full.csv"

# Output 1: unique companies
UNIQUE_COMPANIES = r"C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 4\unique_companies.csv"

# Output 2: unique companies + CIK
UNIQUE_WITH_CIK = r"C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 4\unique_companies_with_cik.csv"

# Standardized schema for parent–subsidiary relationships
PARENT_SUB_FIELDS = [
    "parent_name",
    "parent_cik",
    "subsidiary_name_raw",
    "subsidiary_name_clean",
    "source_type",        # "exhibit_21" for this stage
    "source_url",
    "confidence",         # 0–1
    "extraction_notes",
]

def empty_parent_df(parent_name: str, parent_cik) -> pd.DataFrame:
    """
    Return an empty DataFrame with the standard schema for a given parent.
    Useful when no Exhibit 21 or subsidiaries are found.
    """
    return pd.DataFrame(columns=PARENT_SUB_FIELDS)

# Load normalized company list

In [3]:
companies_df = pd.read_csv(COMPANY_FILE)

# Inspect columns to make sure we use the right ones
companies_df.head()

,company_name
0,1 800 Flowers Com Inc
1,10X Genomics Inc
2,1606 Corp
3,1895 Bancorp Of Wisconsin Inc
4,1stdibs.com Inc


# Load SEC Ticker Dataset (CIK Source)

In [4]:
SEC_LIST_URL = "https://www.sec.gov/files/company_tickers.json"

resp = requests.get(SEC_LIST_URL, headers=SEC_HEADERS)

if resp.status_code != 200:
    print("SEC returned error:", resp.status_code)
    print(resp.text[:300])
    raise SystemExit("Could not load SEC ticker file.")

data = resp.json()

sec_df = pd.DataFrame.from_dict(data, orient='index')

# Normalize titles for matching
sec_df["title_clean"] = (
    sec_df["title"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace("&", "and")
    .str.strip()
)

# Zero-pad CIKs
sec_df["cik_str"] = sec_df["cik_str"].astype(str).str.zfill(10)

print("Loaded SEC companies:", len(sec_df))
sec_df.head()


Loaded SEC companies: 10196


,cik_str,ticker,title,title_clean
0,0001045810,NVDA,NVIDIA CORP,nvidia corp
1,0000320193,AAPL,Apple Inc.,apple inc
2,0001652044,GOOGL,Alphabet Inc.,alphabet inc
3,0000789019,MSFT,MICROSOFT CORP,microsoft corp
4,0001018724,AMZN,AMAZON COM INC,amazon com inc


# SEC Search API for CIKs

In [5]:
def clean_name(name):
    """Normalize company names for matching."""
    return (
        str(name)
        .lower()
        .replace("&", "and")
        .replace(",", "")
        .replace(".", "")
        .strip()
    )

def get_cik(company_name):
    clean = clean_name(company_name)

    # Convert titles to list for stable fuzzy matching
    choices = sec_df["title_clean"].tolist()

    match, score, idx = process.extractOne(
        clean,
        choices,
        scorer=fuzz.WRatio
    )

    # Accept only high-quality matches
    if score >= 85:
        return sec_df.iloc[idx]["cik_str"]

    return None

# Apply CIK Lookup to All Companies

In [6]:
unique_df = pd.read_csv(UNIQUE_COMPANIES)

# If checkpoint exists, resume
if os.path.exists(UNIQUE_WITH_CIK):
    print("Resuming from checkpoint:", UNIQUE_WITH_CIK)
    unique_df = pd.read_csv(UNIQUE_WITH_CIK)

# Ensure CIK column exists
if "cik" not in unique_df.columns:
    unique_df["cik"] = None

total = len(unique_df)

# Start where cik is null
start_index = unique_df[unique_df["cik"].isna()].index.min()
if pd.isna(start_index):
    start_index = 0

print(f"Starting CIK lookup at index {start_index}/{total}")

for i in tqdm(range(start_index, total), desc="Matching CIKs"):
    
    # Skip already matched
    if pd.notna(unique_df.at[i, "cik"]):
        continue

    name = unique_df.at[i, "company_name"]
    cik = get_cik(name)
    
    unique_df.at[i, "cik"] = cik

    # Checkpoint every 100
    if i > 0 and i % 100 == 0:
        unique_df.to_csv(UNIQUE_WITH_CIK, index=False)
        print(f"Checkpoint saved at {i}/{total}")

    # Avoid rate limits
    time.sleep(0.2)

# Final save
unique_df.to_csv(UNIQUE_WITH_CIK, index=False)
print("Completed. Final file saved:", UNIQUE_WITH_CIK)

unique_df.head()


Resuming from checkpoint: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 4\unique_companies_with_cik.csv
Starting CIK lookup at index 623/4629


Matching CIKs:   0%|          | 0/4006 [00:00<?, ?it/s]

Completed. Final file saved: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 4\unique_companies_with_cik.csv


,company_name,cik
0,1 800 Flowers Com Inc,1084869.0
1,10X Genomics Inc,1770787.0
2,1606 Corp,1877461.0
3,1895 Bancorp Of Wisconsin Inc,1847360.0
4,1stdibs.com Inc,1600641.0


# Load New Dataset

In [7]:
companies_df = pd.read_csv(
    "unique_companies_with_cik.csv",
    dtype={"cik": str}   # keep CIK as string, not float
)
companies_df["cik"] = (
    companies_df["cik"]
    .astype(str)
    .str.replace(".0", "", regex=False)   # remove float suffix
    .str.strip()
)

companies_df.head()
companies_df.columns


Index(['company_name', 'cik'], dtype='object')

# Subsidiary name normalization

In [8]:
def normalize_subsidiary_name(name):
    """
    Basic normalization for subsidiary names.
    We are NOT touching parent_name here, since that's already normalized upstream.
    """
    if not isinstance(name, str):
        return None

    n = name.lower().strip()

    # Remove corporate suffixes
    n = re.sub(r'\b(inc|inc\.|llc|l\.l\.c\.|corp|corporation|ltd|limited|company|co)\b', '', n)

    # Remove punctuation
    n = re.sub(r'[.,]', '', n)

    # Remove multiple spaces
    n = re.sub(r'\s+', ' ', n)

    return n.strip()

# Get latest 10-K metadata for a given CIK

In [9]:
def get_latest_10k_metadata(cik):
    url = f"https://data.sec.gov/submissions/CIK{int(cik):010d}.json"
    r = requests.get(url, headers=SEC_HEADERS)
    if not r.ok:
        return None

    data = r.json()
    forms = data.get("filings", {}).get("recent", {})

    for form, acc, primary in zip(
        forms.get("form", []),
        forms.get("accessionNumber", []),
        forms.get("primaryDocument", []),
    ):
        if form == "10-K":
            return {
                "accession": acc.replace("-", ""),
                "primary_doc": primary,
                "cik": cik,
            }
    return None


# Find Exhibit 21 URL for a given 10-K

In [10]:
def find_exhibit_21(cik, accession):
    base = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/"
    index_url = base + "index.json"

    r = requests.get(index_url, headers=SEC_HEADERS)
    if not r.ok:
        return None

    data = r.json()
    items = data.get("directory", {}).get("item", [])

    for file in items:
        name = file["name"].lower()
        if "ex21" in name or "ex-21" in name:
            return base + file["name"]

    return None

# Parse Exhibit 21 to extract subsidiary names

In [11]:
def parse_exhibit_21(url):
    r = requests.get(url, headers=SEC_HEADERS)
    if not r.ok:
        return []

    soup = BeautifulSoup(r.text, "html.parser")

    subs = []

    # Extract from tables first
    tables = soup.find_all("table")
    for table in tables:
        for row in table.find_all("tr"):
            cells = row.find_all(["td", "th"])
            if not cells:
                continue

            text = cells[0].get_text(" ", strip=True)

            # Skip header-like entries
            if text.lower() in ["name", "subsidiary", "entity", "company"]:
                continue

            # Skip junk rows
            if len(text) < 3:
                continue

            # Must contain an indicator of corporate entity
            if not re.search(r'\b(inc|llc|ltd|corp|co|company|bank|group)\b', text, re.I):
                continue

            subs.append(text)

    if subs:
        return subs

    # Fallback: raw text parsing
    text = soup.get_text("\n")
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    pattern = re.compile(r"\b(inc|llc|ltd|corp|company|co|bank|group)\b", re.I)

    clean = [line for line in lines if pattern.search(line)]
    
    return clean


# High-level extraction for Exhibit 21 for a single parent

In [12]:
def extract_exhibit21_subsidiaries(parent_name, cik):
    meta = get_latest_10k_metadata(cik)
    if not meta:
        return empty_parent_df(parent_name, cik)

    ex21_url = find_exhibit_21(cik, meta["accession"])
    if not ex21_url:
        return empty_parent_df(parent_name, cik)

    raw_list = parse_exhibit_21(ex21_url)
    if not raw_list:
        return empty_parent_df(parent_name, cik)

    rows = []
    for raw in raw_list:
        rows.append({
            "parent_name": parent_name,
            "parent_cik": cik,
            "subsidiary_name_raw": raw,
            "subsidiary_name_clean": normalize_subsidiary_name(raw),
            "source_type": "exhibit_21",
            "source_url": ex21_url,
            "confidence": 1.0,
            "extraction_notes": "",
        })

    return pd.DataFrame(rows)


# Controller to run for multiple companies

In [13]:
def build_exhibit21_seed_table(companies):
    results = []
    for i, row in companies.iterrows():
        parent = row["company_name"]
        cik = row["cik"]
        print(f"[{i+1}/{len(companies)}] {parent} ({cik})")

        df_subs = extract_exhibit21_subsidiaries(parent, cik)
        if not df_subs.empty:
            results.append(df_subs)

    if results:
        return pd.concat(results, ignore_index=True)
    else:
        return pd.DataFrame(columns=PARENT_SUB_FIELDS)


# Run for a small test batch

In [14]:
test = companies_df.head(5)
subs_seed = build_exhibit21_seed_table(test)
subs_seed


[1/5] 1 800 Flowers Com Inc (1084869)
[2/5] 10X Genomics Inc (1770787)
[3/5] 1606 Corp (1877461)
[4/5] 1895 Bancorp Of Wisconsin Inc (1847360)
[5/5] 1stdibs.com Inc (1600641)


,parent_name,parent_cik,subsidiary_name_raw,subsidiary_name_clean,source_type,source_url,confidence,extraction_notes
0,1895 Bancorp Of Wisconsin Inc,1847360,"PyraMax Bank, FSB",pyramax bank fsb,exhibit_21,https://www.sec.gov/Archives/edgar/data/184736...,1.0,
1,1895 Bancorp Of Wisconsin Inc,1847360,PyraMax Insurance Services LLC (a wholly owned...,pyramax insurance services (a wholly owned sub...,exhibit_21,https://www.sec.gov/Archives/edgar/data/184736...,1.0,
2,1stdibs.com Inc,1600641,"1stdibs.com, Ltd",1stdibscom,exhibit_21,https://www.sec.gov/Archives/edgar/data/160064...,1.0,


# Full Extractor


In [15]:
CHECKPOINT_FILE = "ex21_checkpoint.csv"
FINAL_FILE = "ex21_full_output.csv"
ERROR_FILE = "ex21_errors.csv"
CHECKPOINT_EVERY = 250  # save every 250 companies

def extract_all_ex21(companies_df):
    results = []
    errors = []

    # Track which CIKs already processed
    processed = set()

    # Resume from checkpoint if exists
    if os.path.exists(CHECKPOINT_FILE):
        cp = pd.read_csv(CHECKPOINT_FILE, dtype=str)
        if not cp.empty:
            results.append(cp)
            processed.update(cp["parent_cik"].astype(str).unique())
            print(f"Loaded checkpoint with {len(processed)} companies already processed.")

    if os.path.exists(ERROR_FILE):
        err_df = pd.read_csv(ERROR_FILE, dtype=str)
        errors = err_df.to_dict("records")
        print(f"Loaded {len(errors)} previous errors.")

    counter = 0
    total = len(companies_df)

    for row in tqdm(companies_df.itertuples(index=False), total=total, desc="Exhibit-21 Extraction"):
        parent = row.company_name
        cik = str(row.cik).replace(".0", "").strip()

        if cik in processed:
            continue

        try:
            df_subs = extract_exhibit21_subsidiaries(parent, cik)
            if not df_subs.empty:
                results.append(df_subs)
        except Exception as e:
            errors.append({
                "parent_name": parent,
                "parent_cik": cik,
                "error": repr(e)
            })

        processed.add(cik)
        counter += 1

        # Save checkpoint every 250 companies
        if counter % CHECKPOINT_EVERY == 0:
            if results:
                pd.concat(results, ignore_index=True).to_csv(CHECKPOINT_FILE, index=False)
                print(f"\nCheckpoint saved — {len(processed)}/{total} companies processed.")
            if errors:
                pd.DataFrame(errors).to_csv(ERROR_FILE, index=False)

        # Stay below SEC rate limits
        time.sleep(0.25)

    # Final save
    full_df = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
    full_df.to_csv(FINAL_FILE, index=False)
    print(f"\nFinal output saved to: {FINAL_FILE}")
    print(f"Total subsidiaries extracted: {len(full_df)}")

    if errors:
        pd.DataFrame(errors).to_csv(ERROR_FILE, index=False)
        print(f"Errors logged to: {ERROR_FILE} ({len(errors)} errors)")

    return full_df

In [16]:
subs_all = extract_all_ex21(companies_df)
subs_all.head()

Exhibit-21 Extraction:   0%|          | 0/4629 [00:00<?, ?it/s]


Checkpoint saved — 250/4629 companies processed.

Checkpoint saved — 500/4629 companies processed.

Checkpoint saved — 750/4629 companies processed.

Checkpoint saved — 1000/4629 companies processed.

Checkpoint saved — 1250/4629 companies processed.

Checkpoint saved — 1500/4629 companies processed.

Checkpoint saved — 1750/4629 companies processed.

Checkpoint saved — 2000/4629 companies processed.

Checkpoint saved — 2250/4629 companies processed.

Checkpoint saved — 2500/4629 companies processed.

Checkpoint saved — 2750/4629 companies processed.

Checkpoint saved — 3000/4629 companies processed.

Checkpoint saved — 3250/4629 companies processed.

Checkpoint saved — 3500/4629 companies processed.

Checkpoint saved — 3750/4629 companies processed.

Checkpoint saved — 4000/4629 companies processed.

Final output saved to: ex21_full_output.csv
Total subsidiaries extracted: 71877
Errors logged to: ex21_errors.csv (1 errors)


,parent_name,parent_cik,subsidiary_name_raw,subsidiary_name_clean,source_type,source_url,confidence,extraction_notes
0,1895 Bancorp Of Wisconsin Inc,1847360,"PyraMax Bank, FSB",pyramax bank fsb,exhibit_21,https://www.sec.gov/Archives/edgar/data/184736...,1.0,
1,1895 Bancorp Of Wisconsin Inc,1847360,PyraMax Insurance Services LLC (a wholly owned...,pyramax insurance services (a wholly owned sub...,exhibit_21,https://www.sec.gov/Archives/edgar/data/184736...,1.0,
2,1stdibs.com Inc,1600641,"1stdibs.com, Ltd",1stdibscom,exhibit_21,https://www.sec.gov/Archives/edgar/data/160064...,1.0,
3,22Nd Century Group Inc,1347858,"22nd Century Limited, LLC",22nd century,exhibit_21,https://www.sec.gov/Archives/edgar/data/134785...,1.0,
4,22Nd Century Group Inc,1347858,"Goodrich Tobacco Company, LLC",goodrich tobacco,exhibit_21,https://www.sec.gov/Archives/edgar/data/134785...,1.0,


# LOGs

In [17]:
def check_ex21_availability(companies_df):
    results = []

    for row in tqdm(companies_df.itertuples(index=False), total=len(companies_df), desc="Checking Exhibit 21"):
        parent = row.company_name
        cik = str(row.cik).replace(".0", "").strip()

        # Default values
        has_ex21 = False
        ex21_url = None
        subsidiary_count = 0

        try:
            # 1. Get 10-K metadata (if none, company has no 10-K)
            meta = get_latest_10k_metadata(cik)
            if not meta:
                results.append({
                    "company_name": parent,
                    "cik": cik,
                    "has_exhibit21": False,
                    "exhibit21_url": None,
                    "subsidiary_count": 0,
                    "reason": "No 10-K found"
                })
                continue

            # 2. Find Exhibit-21 inside that 10-K
            ex21_url = find_exhibit_21(cik, meta["accession"])
            if ex21_url is None:
                results.append({
                    "company_name": parent,
                    "cik": cik,
                    "has_exhibit21": False,
                    "exhibit21_url": None,
                    "subsidiary_count": 0,
                    "reason": "10-K found, but Exhibit-21 missing"
                })
                continue

            # 3. Parse the exhibit to count subsidiaries
            raw_list = parse_exhibit_21(ex21_url)
            subsidiary_count = len(raw_list)
            has_ex21 = True

            results.append({
                "company_name": parent,
                "cik": cik,
                "has_exhibit21": True,
                "exhibit21_url": ex21_url,
                "subsidiary_count": subsidiary_count,
                "reason": "Exhibit-21 found"
            })

        except Exception as e:
            results.append({
                "company_name": parent,
                "cik": cik,
                "has_exhibit21": False,
                "exhibit21_url": None,
                "subsidiary_count": 0,
                "reason": f"Error: {repr(e)}"
            })

        time.sleep(0.25)  # SEC-safe pacing

    return pd.DataFrame(results)


In [18]:
ex21_log = check_ex21_availability(companies_df)
ex21_log.to_csv("companies_exhibit21_log.csv", index=False)
ex21_log.head()


Checking Exhibit 21:   0%|          | 0/4629 [00:00<?, ?it/s]

,company_name,cik,has_exhibit21,exhibit21_url,subsidiary_count,reason
0,1 800 Flowers Com Inc,1084869,False,None,0,"10-K found, but Exhibit-21 missing"
1,10X Genomics Inc,1770787,False,None,0,"10-K found, but Exhibit-21 missing"
2,1606 Corp,1877461,False,None,0,"10-K found, but Exhibit-21 missing"
3,1895 Bancorp Of Wisconsin Inc,1847360,True,https://www.sec.gov/Archives/edgar/data/184736...,2,Exhibit-21 found
4,1stdibs.com Inc,1600641,True,https://www.sec.gov/Archives/edgar/data/160064...,1,Exhibit-21 found


# Merge

In [19]:
import pandas as pd

# 1. Load the Exhibit-21 availability log
log_file = "companies_exhibit21_log.csv"  # output of check_ex21_availability()
ex21_log = pd.read_csv(log_file, dtype=str)

# 2. Load the full extracted subsidiaries dataset
subs_file = "ex21_full_output.csv"  # output of extract_all_ex21()
subs_all = pd.read_csv(subs_file, dtype=str)

# Ensure consistent column names
subs_all = subs_all.rename(columns={
    "parent_name": "company_name",
    "parent_cik": "cik"
})

# 3. Merge the log with the subsidiary list
master = ex21_log.merge(
    subs_all,
    on=["company_name", "cik"],
    how="left"
)

# 4. Fill missing subsidiary names for companies that do not have Exhibit-21
master["subsidiary_name_raw"] = master["subsidiary_name_raw"].fillna("")
master["subsidiary_name_clean"] = master["subsidiary_name_clean"].fillna("")

# 5. Sort for readability
master = master.sort_values(["has_exhibit21", "company_name"], ascending=[False, True])

# 6. Save final master file
master_file = "master_company_subsidiary_dataset.csv"
master.to_csv(master_file, index=False)

master.head()


,company_name,cik,has_exhibit21,exhibit21_url,subsidiary_count,reason,subsidiary_name_raw,subsidiary_name_clean,source_type,source_url,confidence,extraction_notes
3,1895 Bancorp Of Wisconsin Inc,1847360,True,https://www.sec.gov/Archives/edgar/data/184736...,2,Exhibit-21 found,"PyraMax Bank, FSB",pyramax bank fsb,exhibit_21,https://www.sec.gov/Archives/edgar/data/184736...,1.0,NaN
4,1895 Bancorp Of Wisconsin Inc,1847360,True,https://www.sec.gov/Archives/edgar/data/184736...,2,Exhibit-21 found,PyraMax Insurance Services LLC (a wholly owned...,pyramax insurance services (a wholly owned sub...,exhibit_21,https://www.sec.gov/Archives/edgar/data/184736...,1.0,NaN
5,1stdibs.com Inc,1600641,True,https://www.sec.gov/Archives/edgar/data/160064...,1,Exhibit-21 found,"1stdibs.com, Ltd",1stdibscom,exhibit_21,https://www.sec.gov/Archives/edgar/data/160064...,1.0,NaN
7,22Nd Century Group Inc,1347858,True,https://www.sec.gov/Archives/edgar/data/134785...,13,Exhibit-21 found,"22nd Century Limited, LLC",22nd century,exhibit_21,https://www.sec.gov/Archives/edgar/data/134785...,1.0,NaN
8,22Nd Century Group Inc,1347858,True,https://www.sec.gov/Archives/edgar/data/134785...,13,Exhibit-21 found,"Goodrich Tobacco Company, LLC",goodrich tobacco,exhibit_21,https://www.sec.gov/Archives/edgar/data/134785...,1.0,NaN
